In [ ]:
!pip install transformers

# **text summarizing . it might be needed later**

In [ ]:

from transformers import BartForConditionalGeneration, BartTokenizer

model_name = "facebook/bart-large-cnn"
tokenizer = BartTokenizer.from_pretrained(model_name)
model = BartForConditionalGeneration.from_pretrained(model_name)

text = input("paste the text here: ")

inputs = tokenizer([text], max_length=1024, return_tensors='pt', truncation=True)


summary_ids = model.generate(inputs['input_ids'], num_beams=4, max_length=150, early_stopping=True)
summary = tokenizer.decode(summary_ids[0], skip_special_tokens=True)

print(summary)

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

paste the text here: *The Coffee That Scares Me*  I don't count my coffees anymore. Counting would make it real.  It starts innocent. One cup in the morning, because how else is a human supposed to become a functional person? The steam, that first bitter sip - it feels like someone switched the lights on in my brain.  Then comes the second cup. Not because I need it, but because the first one was so good, and I have work to do, and emails, and life. The second cup is for productivity.  The third cup is where the justification starts. "Today is a long day." "I slept late." "Just this one more." By the third cup, my hands are a little faster, my heart a little louder.  And then there's the fourth one. The one I drink even though I know I shouldn't. The one that makes me say at night, "the amount of coffee I consume scares me too 🙂" That smiley is important. It's how we say, "I know this is not healthy, but please don't lecture me."  The scary part is not the coffee. It's what it reveals.

# **testing the gpu**

In [ ]:
import torch
print(torch.cuda.is_available())
print(torch.__version__)
!nvidia-smi

True
2.11.0+cu128
Thu Sep 17 11:21:04 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   31C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------

#**make sure the runtime is aved to T4 gpu**

In [ ]:
!wget -O /dev/null https://huggingface.co/Qwen/Qwen3-8B/resolve/main/model-00001-of-00004.safetensors 2>&1 | tail -5

Resolving huggingface.co (huggingface.co)... 99.86.101.39, 99.86.101.56, 99.86.101.64, ...
Connecting to huggingface.co (huggingface.co)|99.86.101.39|:443... connected.
HTTP request sent, awaiting response... 404 Not Found
2026-09-17 11:22:18 ERROR 404: Not Found.



#**importing hugging face transformers**

In [ ]:
import time
start = time.time()

!pip install -U bitsandbytes
import bitsandbytes
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_compute_dtype=torch.float16)

t0 = time.time()
tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen3-8B")
print(f"Tokenizer loaded in {time.time()-t0:.1f}s")

t1 = time.time()
model1 = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen3-8B",
    quantization_config=bnb_config,
    device_map="auto"
)
print(f"Model loaded in {time.time()-t1:.1f}s")
print(f"Total: {time.time()-start:.1f}s")

Tokenizer loaded in 1.5s


Loading weights:   0%|          | 0/399 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Model loaded in 68.1s
Total: 81.0s


#**testing for a simple text....if it takes more than 5 mins**
#**terminate it and make sure all other sessions of the colab is terminated**

In [ ]:
messages = [
    {"role": "user", "content": "Tell me about coffee"}
]

tokenized_inputs = tokenizer.apply_chat_template(
    messages,
    tokenize=True,
    add_generation_prompt=True,
    return_tensors="pt",
    return_dict=True
)

input_ids_on_device = tokenized_inputs['input_ids'].to(model1.device)
attention_mask_on_device = tokenized_inputs['attention_mask'].to(model1.device)

output = model1.generate(
    input_ids=input_ids_on_device,
    attention_mask=attention_mask_on_device,
    max_new_tokens=100
)


generated_tokens = output[0][input_ids_on_device.shape[-1]:]
print(tokenizer.decode(generated_tokens, skip_special_tokens=True))

<think>
Okay, the user asked me to tell them about coffee. Let me start by thinking about what they might need. They could be a coffee enthusiast looking for more information, or maybe someone new to coffee who wants a basic overview. I should cover the basics but also include some depth.

First, I should mention the origin of coffee. Where did it come from? Ethiopia is the birthplace, right? The story of Kaldi and the goats is a common legend, but I should note


In [ ]:
from transformers import BartForConditionalGeneration, BartTokenizer

bart_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")
bart_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")

while True:
  user = __builtins__.input("Enter your name: ")
  messages = [{"role": "user", "content": user}]
  tokenized_inputs = tokenizer.apply_chat_template(
      messages,
      tokenize=True,
      add_generation_prompt=True,
      return_tensors="pt",
      return_dict=True)
  input_ids_on_device = tokenized_inputs['input_ids'].to(model1.device)
  attention_mask_on_device = tokenized_inputs['attention_mask'].to(model1.device)
  output = model1.generate(
    input_ids=input_ids_on_device,
    attention_mask=attention_mask_on_device,
    max_new_tokens=100
    )
  generated_tokens = output[0][input_ids_on_device.shape[-1]:]
  print(tokenizer.decode(generated_tokens, skip_special_tokens=True))

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

KeyboardInterrupt: Interrupted by user